In [1]:
import pandas as pd
import numpy as np

In [2]:
class FX:
    
    #Risk weight table for Delta
    risk_weight = 0.15
    preferential_risk_weight = risk_weight / np.sqrt(2)
    
    def __init__(self, data, extra_prescribed_currency=None):
        #Assume the data has already been cleansed, filtered and grouped
        self.original_data = data
        self.data = data.copy()
        self.prescribed_currencies = ["AUD", "BRL", "CAD", "CHF", "CNY", "EUR", "GBP", "HKD", "INR", "JPY", "KRW", "MXN", "NOK", "NZD", "RUB", "SEK", "SGD", "TRY", "USD", "ZAR"] #These are the currencies that enjoy lower risk weights by scale of 1/sqrt(2)
        if extra_prescribed_currency != None and extra_prescribed_currency not in self.prescribed_currencies:
            self.prescribed_currencies.append(extra_prescribed_currency)
        
        
    def weight_sensitivities(self):
        self.data["Weighted Sensitivities"] = self.data["Sensitivity (reporting currency equiv.)"] * np.select([self.data["Sensi Type"] == "Delta"], [self.data["SA Bucket"].isin(self.prescribed_currencies).map({True: FX.preferential_risk_weight, False: FX.risk_weight})], default=1)

    def perform_intra_bucket_aggregation(self, sensi_type):
        #Filter data to only include targeted sensitivity type
        filtered_data = self.data[self.data["Sensi Type"] == sensi_type]
        
        #Aggregate data of the same sensitivities
        if sensi_type == "Delta":
            cols_to_grp = ["SA Bucket"]
        elif sensi_type == "Vega":
            cols_to_grp = ["SA Bucket", "Tenor"]
        else:
            cols_to_grp = ["SA Bucket", "CVR+/CVR-"]
        
        filtered_data = filtered_data.groupby(cols_to_grp)["Weighted Sensitivities"].sum().reset_index()
        
        #Create a list of unique buckets
        bucket_list = filtered_data["SA Bucket"].drop_duplicates()
        
        #Initiate variables to store K for each bucket
        temp_Kb_by_bucket = [[], [], []]   #[[medium], [high], [low]]
        K_by_bucket = []
        
        if sensi_type == "Delta":
            
            for i in range(3):
                temp_Kb_by_bucket[i].append(np.sqrt(np.maximum(0, filtered_data["Weighted Sensitivities"] ** 2)))
                K_by_bucket.append(pd.DataFrame({"SA Bucket": bucket_list, "K": temp_Kb_by_bucket[i][0]}))

        elif sensi_type == "Vega":
            
            #Initiate variables to store correlation matrix for each bucket
            corr_matrix_by_bucket = dict.fromkeys(bucket_list, None)
            
            for bucket in bucket_list:
                #Filter data for each bucket
                bucket_data = filtered_data[filtered_data["SA Bucket"] == bucket].reset_index(drop=True)
                weighted_sensi = np.asarray(bucket_data["Weighted Sensitivities"])
                
                n = len(bucket_data)
                temp_corr_matrix = np.identity(n)
                
                for i in range(n):
                    for j in range(n):
                        if i != j:
                            #Calculate correlation based on option tenor under medium scenario
                            temp_corr = np.exp(
                                        -0.01 * np.abs(bucket_data.loc[i, "Tenor"] - bucket_data.loc[j, "Tenor"]) / 
                                        np.minimum(bucket_data.loc[i, "Tenor"], bucket_data.loc[j, "Tenor"]))
                            temp_corr_matrix[i, j] = np.minimum(temp_corr, 1)
                
                #High and low scenario
                temp_high_corr_matrix = np.minimum(1.25 * temp_corr_matrix, 1)
                temp_low_corr_matrix = np.maximum(2 * temp_corr_matrix - 1, 0.75 * temp_corr_matrix)
                
                #Store results
                corr_matrix_by_bucket[bucket] = [temp_corr_matrix, temp_high_corr_matrix, temp_low_corr_matrix]
                
                #Calculate and store result of Kb for each bucket 
                for i in range(3):
                    temp_Kb_by_bucket[i].append(np.sqrt(np.maximum(0, np.dot(np.transpose(weighted_sensi),np.dot(weighted_sensi, corr_matrix_by_bucket[bucket][i])))))
                
            for i in range(3):
                K_by_bucket.append(pd.DataFrame({"SA Bucket": bucket_list, "K": temp_Kb_by_bucket[i]}).reset_index(drop=True))
        
        else:
            
            #Compare whether upward or downward scenarios are applicable and calculate their values
            K_by_bucket = filtered_data.pivot_table(index="SA Bucket", columns="CVR+/CVR-", values="Weighted Sensitivities", aggfunc="sum").reset_index()
            K_by_bucket["K+"] = np.sqrt(np.maximum(K_by_bucket["CVR+"], 0) ** 2)
            K_by_bucket["K-"] = np.sqrt(np.maximum(K_by_bucket["CVR-"], 0) ** 2)
            K_by_bucket["K"] = np.maximum(K_by_bucket["K+"], K_by_bucket["K-"])
            
        return K_by_bucket
    
    def perform_across_bucket_aggregation(self, sensi_type, intra_agg_result):
        
        capital_charge = []
        
        if sensi_type != "CVR":
            
            filtered_data = self.data[self.data["Sensi Type"] == sensi_type]
            
            #Calculate Sb according to formulation in MAR21.4(5)
            Sb_by_bucket = filtered_data.groupby("SA Bucket")["Weighted Sensitivities"].sum().reset_index()
            
            #Generate the correlation matrix
            n = len(Sb_by_bucket)
            medium_corr_matrix = np.full((n, n), 0.6)
            np.fill_diagonal(medium_corr_matrix, 0)
            
            high_corr_matrix = np.minimum(1.25 * medium_corr_matrix, 1)
            low_corr_matrix = np.maximum(2 * medium_corr_matrix - 1, 0.75 * medium_corr_matrix)
            
            corr_matrices = [medium_corr_matrix, high_corr_matrix, low_corr_matrix]
                            
            #Calculate the capital charge under each scenario [medium, high, low]
            for i in range(3):
                Kb = intra_agg_result[i]["K"]
                Sb = Sb_by_bucket["Weighted Sensitivities"]
                capital_charge.append(np.sqrt(
                                    np.square(Kb).sum() 
                                    + np.dot(np.transpose(Sb), np.dot(Sb, corr_matrices[i]))
                                    ))
                
                #For each scenario, we have to check whether an alternative formulation for Sb is required as stipulated in MAR21.4(5)
                if capital_charge[i] < 0:
                    Sb_Alt = np.maximum(np.minimum(Sb, Kb), -1 * Kb)
                    capital_charge[i] = (np.sqrt(np.square(Kb).sum() + np.dot(np.transpose(Sb_Alt),np.dot(Sb_Alt, corr_matrices[i]))))
                    
        
        else:
            
            n = len(intra_agg_result)
            
            #Generate the correlation matrix
            medium_corr_matrix = np.full((n, n), 0.6 ** 2)
            np.fill_diagonal(medium_corr_matrix, 0)
            
            high_corr_matrix = np.minimum(1.25 * medium_corr_matrix, 1)
            low_corr_matrix = np.maximum(2 * medium_corr_matrix - 1, 0.75 * medium_corr_matrix)
            
            corr_matrices = [medium_corr_matrix, high_corr_matrix, low_corr_matrix]
            
            Sb = []    
            for i in range(n):                
                #Calculate Sb according to formulation in MAR21.5(4)
                if intra_agg_result["K"][i] == intra_agg_result["K+"][i]:
                    Sb.append(intra_agg_result["CVR+"][i])
                elif intra_agg_result["K"][i] == intra_agg_result["K-"][i]:
                    Sb.append(intra_agg_result["CVR-"][i])
                elif intra_agg_result["CVR+"][i] > intra_agg_result["CVR-"][i]:
                    Sb.append(intra_agg_result["CVR+"][i])
                else:
                    Sb.append(intra_agg_result["CVR-"][i])
            
            #Generate the psi matrix according to MAR21.5(4)(b)
            psi_matrix = np.zeros((n, n))
            for i in range(n):
                for j in range(n):
                    if Sb[i] >= 0 or Sb[j] >= 0:
                        psi_matrix[i, j] = 1
            
            #Store results                
            intra_agg_result["Sb"] = Sb
            
            #Calculate capital charge
            for i in range(3):
                capital_charge.append(np.sqrt(
                                        np.maximum(0, 
                                        np.square(intra_agg_result["K"]).sum() + 
                                        np.dot(Sb, np.dot(np.transpose(Sb), np.multiply(corr_matrices[i], psi_matrix)))
                                        )))        
    
        result = pd.DataFrame({"Medium": [capital_charge[0]], "High": [capital_charge[1]], "Low": [capital_charge[2]]}, index=[sensi_type])
        
        return result

In [ ]:
#For quick test
test = pd.read_excel("Data/temp_frtb_data.xlsx", "FX")

x = FX(test)
x.weight_sensitivities()
intra = x.perform_intra_bucket_aggregation("CVR")
x.perform_across_bucket_aggregation("CVR", intra)
intra

CVR+/CVR-,SA Bucket,CVR+,CVR-,K+,K-,K,Sb
0,CAD,-4000.0,5000.0,0.0,5000.0,5000.0,5000.0
1,USD,-6000.0,7000.0,0.0,7000.0,7000.0,7000.0
